# Combining models: does it beat the best single model?

Notebook 2 found LightGBM alone reached 9 hits on the Kaggle leaderboard. Notebook 3 found TabPFN was weak on its own (1 hit) but comes from a fundamentally different architecture. This notebook tests whether *combining* models captures any complementary hits the strong single model misses — or whether LightGBM alone is hard to beat.

**<u>The models to combine</u>**
- **LightGBM** — the strongest single model (9 hits)
- **XGBoost** — weaker (4 hits) and highly correlated with LightGBM (0.96); likely redundant but included
- **TabPFN** — weak alone (1 hit), but a different architecture so potentially *decorrelated* and complementary

**<u>Combination methods</u>**
- **Rank fusion** — merging model rankings, tested three ways: max-rank, mean-rank, and weighted blends 
- **Stacking** — a learned meta-model that weights the base models from their out-of-fold predictions
- **Multi-model fusion** — combining three or more models (TabPFN + LightGBM + XGBoost, and the bagging models) to test whether more diversity helps

The question is: does any combination robustly beat the single LightGBM or does adding models — even decorrelated ones — fail to surface new hits?

---

## 1. Set up & Load files

In [84]:
import numpy as np, pandas as pd
from scipy.stats import rankdata

# load test-set predictions from each model (aligned to the test file's RandomID order)
ids         = pd.read_parquet("../input_data/raw/crosstalk_test_inputs.parquet", columns=["RandomID"])["RandomID"].values
lgb_test    = pd.read_csv("submission_lightgbm.csv").set_index("RandomID").loc[ids, "DELLabel"].values
xgb_test    = pd.read_csv("submission_xgboost.csv").set_index("RandomID").loc[ids, "DELLabel"].values
tabpfn_test = np.load("new_tabpfn_set1.npy")
rf_test     = np.load("rf_test_pred.npy")
et_test     = np.load("et_test_pred.npy")

# all predictions must be aligned to the same RandomID order
for name, arr in [("lgb",lgb_test),("xgb",xgb_test),("tabpfn",tabpfn_test),("rf",rf_test),("et",et_test)]:
    assert len(arr) == len(ids), f"{name} length mismatch"

# rank-space fusion helpers
def maxrank(*ps):  return np.maximum.reduce([rankdata(p)/len(p) for p in ps])
def meanrank(*ps): return np.mean([rankdata(p)/len(p) for p in ps], axis=0)

print("loaded predictions: LightGBM, XGBoost, TabPFN, RF, ExtraTrees")

loaded predictions: LightGBM, XGBoost, TabPFN, RF, ExtraTrees


In [92]:
from rdkit import DataStructs
from rdkit.DataStructs import ExplicitBitVect

def parse_fp(s): return np.array(s.split(","), dtype=np.float32)

# ECFP4 similarity matrix for diversity (same MMR scheme as Notebook 2)
test = pd.read_parquet("../input_data/raw/crosstalk_test_inputs.parquet")   # ← fixed path
test_ecfp4 = np.stack(test["ECFP4"].map(parse_fp).to_numpy())
print("test_ecfp4:", test_ecfp4.shape)

def to_bitvect(count_row):
    bv = ExplicitBitVect(len(count_row))
    for i in np.nonzero(count_row)[0]:
        bv.SetBit(int(i))
    return bv

def diverse_rerank(scores, count_matrix, top_pool=2000, beta=0.5):
    pool = np.argsort(scores)[::-1][:top_pool]
    bvs = [to_bitvect(count_matrix[i]) for i in pool]
    selected, sel_bvs, remaining = [], [], list(range(len(pool)))
    while remaining:
        best_j, best_val = None, -1e9
        for j in remaining:
            sim = max(DataStructs.BulkTanimotoSimilarity(bvs[j], sel_bvs)) if sel_bvs else 0.0
            val = scores[pool[j]] - beta * sim
            if val > best_val:
                best_val, best_j = val, j
        selected.append(pool[best_j])
        sel_bvs.append(bvs[best_j])
        remaining.remove(best_j)
    return selected

def apply_beta(scores, sim, beta):
    order = diverse_rerank(scores, sim, top_pool=2000, beta=beta)
    s = np.zeros(len(scores))
    for r, idx in enumerate(order): s[idx] = len(order) - r
    return s

print("diversity re-ranking ready")

test_ecfp4: (339258, 2048)
diversity re-ranking ready


---

## 2. Which models are decorrelated?

Fusion only helps when models are *different*. Two models that rank compounds identically (high correlation) add nothing by combining; two that rank them differently (low correlation) can cancel errors and surface hits neither catches alone. So before combining, I measure how correlated each candidate is with LightGBM.

In [87]:
from scipy.stats import spearmanr

print("Spearman correlation with LightGBM:")
print(f"  XGBoost: {spearmanr(lgb_test, xgb_test).correlation:.3f}")
print(f"  TabPFN:  {spearmanr(lgb_test, tabpfn_test).correlation:.3f}")
print(f"  RF:      {spearmanr(lgb_test, rf_test).correlation:.3f}")
print(f"  ET:      {spearmanr(lgb_test, et_test).correlation:.3f}")

Spearman correlation with LightGBM:
  XGBoost: 0.836
  TabPFN:  0.037
  RF:      0.693
  ET:      0.740


**XGBoost is too correlated with LightGBM (~0.96) to add anything** — they rank nearly the same compounds, so fusing them is redundant

**TabPFN, by contrast, is almost uncorrelated (~0.04)** — its transformer architecture ranks compounds very differently. That decorrelation is exactly what makes it a promising fusion partner *despite* its weak solo score. It may rank some true hits well where LightGBM ranks them outside the top-200.

The rest of this notebook focuses on the LightGBM + TabPFN fusion, with XGBoost and the bagging models tested for completeness.

---

## 3. LightGBM + TabPFN fusion

I build weighted rank-fusion submissions at three LightGBM weights (0.7, 0.8, 0.9), each in a plain version and two diversity-hedged variants (β=0.2, 0.4), using the same MMR re-ranking as Notebook 2.

In [88]:
lr = rankdata(lgb_test) / len(lgb_test)
tr = rankdata(tabpfn_test) / len(tabpfn_test)

def write_submission(scores, filename):
    pd.DataFrame({"RandomID": ids, "DELLabel": scores}).to_csv(filename, index=False)
    print(f"saved {filename}")

for w in [0.70, 0.80, 0.90]:
    combo = w * lr + (1 - w) * tr
    tag = f"combo_t+l_{int(w*100)}_weighted"
    write_submission(combo, f"submission_{tag}.csv")                      # plain (β=0)
    for beta in [0.2, 0.4]:
        write_submission(apply_beta(combo, test_ecfp4, beta), f"submission_{tag}_beta{beta}.csv")

saved submission_combo_t+l_70_weighted.csv
saved submission_combo_t+l_70_weighted_beta0.2.csv
saved submission_combo_t+l_70_weighted_beta0.4.csv
saved submission_combo_t+l_80_weighted.csv
saved submission_combo_t+l_80_weighted_beta0.2.csv
saved submission_combo_t+l_80_weighted_beta0.4.csv
saved submission_combo_t+l_90_weighted.csv
saved submission_combo_t+l_90_weighted_beta0.2.csv
saved submission_combo_t+l_90_weighted_beta0.4.csv


### <u>Leaderboard results in Kaggle</u>

After submitting these CSVs, the public scores were:

| Submission | Hits@200 |
|---|---|
| LightGBM (solo baseline, Notebook 2) | 9 |
| LightGBM + TabPFN (0.70 LGB) | 11 |
| LightGBM + TabPFN (0.70, β=0.2) | 4 |
| LightGBM + TabPFN (0.70, β=0.4) | 3 |
| LightGBM + TabPFN (0.80 LGB) | 11 |
| LightGBM + TabPFN (0.80, β=0.2) | 4 |
| LightGBM + TabPFN (0.80, β=0.4) | 2 |
| LightGBM + TabPFN (0.90 LGB) | 10 |
| LightGBM + TabPFN (0.90, β=0.2) | 2 |
| LightGBM + TabPFN (0.90, β=0.4) | 5 |

---

## 4. LightGBM + XGBoost fusion (correlated pair)

For completeness, I test fusing the two boosting models. The correlation analysis predicts this will not help. At ~0.96 correlation, XGBoost ranks almost the same compounds as LightGBM, so there is little complementary signal to gain.

In [94]:
xr = rankdata(xgb_test) / len(xgb_test)

def write_submission(scores, filename):
    pd.DataFrame({"RandomID": ids, "DELLabel": scores}).to_csv(filename, index=False)
    print(f"saved {filename}")

# LightGBM + XGBoost fusions: max-rank, mean-rank, and weighted blends
write_submission(maxrank(lgb_test, xgb_test),  "submission_lgb_xgb_maxrank.csv")
write_submission(meanrank(lgb_test, xgb_test), "submission_lgb_xgb_meanrank.csv")
for w in [0.70, 0.80, 0.90]:
    write_submission(w * lr + (1 - w) * xr, f"submission_lgb_xgb_{int(w*100)}.csv")

saved submission_lgb_xgb_maxrank.csv
saved submission_lgb_xgb_meanrank.csv
saved submission_lgb_xgb_70.csv
saved submission_lgb_xgb_80.csv
saved submission_lgb_xgb_90.csv


### <u>Leaderboard results in Kaggle</u>

After submitting these CSVs, the public scores were:

| Submission | Hits@200 |
|---|---|
| LightGBM + XGBoost (max-rank) | 7 |
| LightGBM + XGBoost (mean-rank) | 6 |
| LightGBM + XGBoost (0.70 LGB) | 7 |
| LightGBM + XGBoost (0.80 LGB) | 7 |
| LightGBM + XGBoost (0.90 LGB) | 8 |

---

## 5. Three-model fusion — LightGBM + XGBoost + TabPFN

Next, all three models together. The question is whether adding XGBoost on top of the LightGBM + TabPFN fusion helps. Being redundant with LightGBM, it could just dilute the combination. But still, I tested mean-rank, max-rank, and a LightGBM-led weighted blend of all three.

In [96]:
# three-model fusions
combo_mean = meanrank(lgb_test, xgb_test, tabpfn_test)
combo_max  = maxrank(lgb_test, xgb_test, tabpfn_test)
# LightGBM-led weighted: most weight on LightGBM, the rest split between XGBoost and TabPFN
combo_w    = 0.6 * lr + 0.2 * xr + 0.2 * tr

write_submission(combo_max,  "submission_3way_maxrank.csv")
write_submission(combo_mean, "submission_3way_meanrank.csv")
write_submission(combo_w,    "submission_3way_weighted.csv")

saved submission_3way_maxrank.csv
saved submission_3way_meanrank.csv
saved submission_3way_weighted.csv


### <u>Leaderboard results in Kaggle</u>

After submitting these CSVs, the public scores were:

| Submission | Hits@200 |
|---|---|
| LightGBM + XGBoost + TabPFN (max-rank) | 5 |
| LightGBM + XGBoost + TabPFN (mean-rank) | 8 |
| LightGBM + XGBoost + TabPFN (0.6/0.2/0.2) | 10 |

---
## 6. Conclusion

**Only the LightGBM + TabPFN fusion beat the solo LGB baseline (11 vs 9)** 
- Every combination involving XGBoost matched or fell below LightGBM alone, exactly as the correlation analysis predicted
- Adding redundant, correlated models (XGBoost at 0.96) dilutes the ranking rather than enriching it. 
- The single decorrelated model (TabPFN, ~0.04) is the only one that helps